# 09 Dashboard Layer Preparation

This notebook prepares WebGIS-ready layers and tables for the Streamlit dashboard.

## What you will do

1. Review dashboard input expectations.
2. Export GeoJSON layers from processed vectors and rasters.
3. Copy important CSV tables to dashboard output folders.
4. Inspect exported dashboard layers.
5. Confirm how to run the dashboard.

## Step 1: Load configuration and inspect dashboard folders

In [ ]:
from pathlib import Path
import json
import pandas as pd
from floodsense.config import load_config
from floodsense.paths import list_input_files

config = load_config()
paths = config['paths']
dashboard_layers = Path(paths['dashboard_layers'])
dashboard_tables = Path(paths['outputs_dashboard'])
print('Dashboard layer folder:', dashboard_layers)
print('Dashboard table folder:', dashboard_tables)

## Step 2: Understand dashboard-ready outputs

Expected GeoJSON layers:

- `lokoja_boundary.geojson`
- `flood_susceptibility_zones.geojson`
- `observed_flood_extent.geojson`
- `exposed_buildings.geojson`
- `exposed_roads.geojson`
- `priority_zones.geojson` if available

Expected tables:

- `exposure_summary.csv`
- `validation_metrics.csv`
- `priority_ranking.csv`

## Step 3: Export dashboard layers

This step reads processed outputs and writes portable GeoJSON/CSV files for Streamlit/Folium. Missing layers are skipped with clear messages.

In [ ]:
from floodsense.export import export_dashboard_layers

export_dashboard_layers(config)

## Step 4: Inspect exported GeoJSON layers

In [ ]:
geojson_files = list_input_files(dashboard_layers, ['.geojson', '.json'])
print('Dashboard GeoJSON/JSON files:', len(geojson_files))
for file in geojson_files:
    print('  -', file.name)

## Step 5: Inspect copied dashboard tables

In [ ]:
csv_files = list_input_files(dashboard_tables, ['.csv'])
print('Dashboard CSV files:', len(csv_files))
for file in csv_files:
    print('  -', file.name)
    try:
        display(pd.read_csv(file).head())
    except Exception as exc:
        print('    Could not read table:', exc)

## Step 6: Read metadata file

In [ ]:
metadata_path = dashboard_layers / 'metadata.json'
if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    metadata
else:
    print('metadata.json not available yet.')

## Step 7: Run the dashboard

From the repository root, run:

```bash
streamlit run dashboard/app.py
```

The dashboard will load any available GeoJSON layers and CSV tables. Missing outputs are displayed as unavailable rather than causing the dashboard to fail.

## Final workflow reminder

Recommended full execution order:

```bash
pip install -e .
python scripts/01_prepare_data.py
python scripts/02_run_hydrology.py
python scripts/03_build_susceptibility_model.py
python scripts/04_run_sar_validation.py
python scripts/05_run_exposure_analysis.py
python scripts/06_build_priority_index.py
python scripts/07_export_dashboard_layers.py
streamlit run dashboard/app.py
```

Or run all stages with:

```bash
python scripts/run_pipeline.py --stage all
```